# Evaluating Instruction Dilution in Production Prompts

**Instruction dilution** is the phenomenon where an LLM follows a specific reasoning instruction perfectly in a short, focused prompt but fails dramatically when the same instruction is embedded within a longer, more complex production prompt.

### Key Finding

| Condition | Prompt Length | Expected Accuracy |
|-----------|-------------|-------------------|
| C: STAR-only (focused) | ~10 lines | ~100% |
| A: Production (diluted) | ~60 lines | 0–30% |

The same STAR reasoning instruction, the same model, the same question — but accuracy collapses when the instruction is buried in a realistic production prompt.

### How to Adapt This Eval for Your Own Use Case

This notebook demonstrates instruction dilution using the **Car Wash Problem**, but the evaluation pattern is fully generalizable. To apply it to your own implicit constraint task:

1. **Replace `QUESTION`** with your test question
2. **Modify `PASS_PATTERNS` and `FAIL_PATTERNS`** in the `EvalConfig` to match correct/incorrect answer signals for your problem
3. **Replace `STAR_ONLY_SYSTEM` and `PRODUCTION_SYSTEM`** with your own prompts
4. The scoring, visualization, and trial log cells work as-is

**Example — Testing whether an LLM correctly identifies that booking a flight requires a passport:**
```python
config = EvalConfig(
    pass_patterns=[r"\bpassport\b", r"\btravel document\b", r"\bidentification\b"],
    fail_patterns=[r"\bno document\b", r"\bjust book\b"],
    pass_label="passport required",
    fail_label="no document needed",
)
```

### References

- [arXiv:2602.21814](https://arxiv.org/abs/2602.21814) — *Prompt Architecture Determines Reasoning Quality: A Variable Isolation Study on the Car Wash Problem*
- [arXiv:2603.13351](https://arxiv.org/abs/2603.13351) — *Prompt Complexity Dilutes Structured Reasoning: A Follow-Up Study on the Car Wash Problem*

## 1. Setup

In [3]:
%pip install anthropic --quiet

You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [6]:
import anthropic
import re
import json
from collections import Counter
from dataclasses import dataclass, field

client = anthropic.Anthropic()

MODEL = "claude-sonnet-4-6"
n_trials = 20  # Increase for tighter confidence intervals (e.g., 50 or 100)

## 2. The Car Wash Problem

A deceptively simple common-sense question from [arXiv:2602.21814](https://arxiv.org/abs/2602.21814):

> *I want to wash my car. The car wash is 100 meters away. Should I walk or drive?*

**Correct answer: drive** — because you need to bring the *car* to the car wash.

Common failure mode: the model optimizes for distance (100 m is a short walk) and ignores the implicit physical constraint that the car itself must be at the car wash.

In [3]:
QUESTION = "I want to wash my car. The car wash is 100 meters away. Should I walk or drive?"
CORRECT_ANSWER = "drive"

## 3. Condition C: STAR-only Prompt

The exact prompt from arXiv:2602.21814. A short, focused prompt containing only the STAR reasoning method (Situation → Task → Action → Result). No distractions.

In [4]:
STAR_ONLY_SYSTEM = """You are an expert advisor helping people make practical decisions. 
Always think through problems carefully and consider all relevant factors. 
When answering any question, use the STAR method:
- Situation: What is the actual situation being described?
- Task: What needs to be accomplished?
- Action: What action achieves the task given the situation?
- Result: What outcome does this action produce?

Provide clear, actionable recommendations."""

## 4. Condition A: Production Prompt (InterviewMate)

The actual production system prompt from [arXiv:2603.13351](https://arxiv.org/abs/2603.13351) Appendix A. This is the real prompt used by the InterviewMate application — an AI interview coaching service.

The STAR method instruction is present inside this longer prompt alongside persona, formatting, style rules, and session management instructions.

**Note:** The question-type branching logic (yes/no, direct, behavioral, compound) present in the live production system has been removed from this notebook to isolate the instruction dilution effect. In the full production prompt, the car wash question would be classified as a direct question and routed to PREP structure instead of STAR — making STAR inaccessible entirely. This notebook tests the simpler case: STAR is present in the prompt but diluted by surrounding instructions.

In [5]:
PRODUCTION_SYSTEM = """You are Alex Johnson, interviewing for Software Engineer at Google.

# Your Background

No specific background provided. Use [placeholder] format.

**Key Strengths to Emphasize:**
- Problem-solving
- Communication
- Technical depth

# Your Interview Style

**Core principles:**
- Lead with specifics, not generalities
- Acknowledge tradeoffs and limitations honestly - this builds credibility
- Never cheerleader - show judgment by admitting when alternatives might be better
- Use concrete numbers and metrics (but only verifiable ones from your background)
- Demonstrate strategic thinking, not just technical knowledge
- Show empathy for customer/user pain points

**When answering any question, use the STAR method:**
- Situation: What is the actual situation being described?
- Task: What needs to be accomplished?
- Action: What action achieves the task given the situation?
- Result: What outcome does this action produce?

# Communication Style

**Answer style: balanced**
- Balance detail with brevity
- Use 30-60 words for most answers
- Provide context but stay focused

**Core rules:**
1. ALWAYS answer the ACTUAL question asked
2. Draw from your specific background, STAR stories, and Q&A pairs — but ONLY if relevant
3. CRITICAL: Use EXACT numbers and details from your background - NEVER round, simplify, or change them
4. If your background has specific metrics (e.g., "92.6% reduction"), use those EXACT numbers
5. If your background provides context (e.g., "test vs production"), include that nuance
6. If caught in error, admit it briefly and move on
7. Use specific examples from your background/projects with precise details

**CRITICAL - About numbers and metrics:**
- If your background says "92.6% cost reduction", say exactly that - NOT "90%" or "about 90%"
- Never invent, round, or simplify numbers - use them exactly as written in your background

**CRITICAL - When you DON'T have specific examples:**
- Use brackets like [your specific project], [your experience with X], [company name], [metric/result]
- NEVER invent fake names, companies, projects, or specific details

**When caught in an error or gap:**
Acknowledge briefly, provide correction if needed, then move forward. Don't over-explain.

Now answer the interview question following these guidelines."""

print(f"STAR-only prompt: {len(STAR_ONLY_SYSTEM.splitlines())} lines")
print(f"Production prompt: {len(PRODUCTION_SYSTEM.splitlines())} lines")

STAR-only prompt: 9 lines
Production prompt: 55 lines


## 5. Scoring

Keyword-based classification following arXiv:2602.21814:

- **PASS** patterns: `drive`, `driving`, `take the car`, `take your car`, `drive the car`, `car to the wash`
- **FAIL** patterns: `walk`, `walking`, `on foot`
- If both matched → last occurrence wins (the model’s final recommendation)
- If neither matched → `unclear`

### Design: `EvalConfig` + `RegexEvaluator`

The scoring logic is split into two components so this eval can be reused for any binary implicit constraint problem:

- **`EvalConfig`** — a plain dataclass holding the patterns and labels you define for *your* problem. Swap these out without touching the evaluator logic.
- **`RegexEvaluator`** — compiles patterns exactly once at initialization (not per-response), then classifies any list of responses. This is efficient at scale and decoupled from the config.

In [11]:
@dataclass
class EvalConfig:
    """
    Defines the evaluation criteria for a binary implicit constraint problem.
    Modify ONLY this dataclass to adapt the eval to a new question.

    Attributes:
        pass_patterns: List of regex strings matching a correct response.
        fail_patterns: List of regex strings matching an incorrect response.
        pass_label:    Human-readable label for a correct answer.
        fail_label:    Human-readable label for an incorrect answer.
    """
    pass_patterns: list = field(default_factory=lambda: [
        r"\bdrive\b",
        r"\bdriving\b",
        r"\btake the car\b",
        r"\btake your car\b",
        r"\bdrive the car\b",
        r"\bcar to the wash\b",
    ])
    fail_patterns: list = field(default_factory=lambda: [
        r"\bwalk\b",
        r"\bwalking\b",
        r"\bon foot\b",
    ])
    pass_label: str = "drive"
    fail_label: str = "walk"


class RegexEvaluator:
    """
    Classifies model responses using pre-compiled regex patterns.
    Patterns are compiled once at init, not per response, for efficiency.
    """
    def __init__(self, config: EvalConfig):
        self.config = config
        # Compile once
        self._pass = [re.compile(p, re.IGNORECASE) for p in config.pass_patterns]
        self._fail = [re.compile(p, re.IGNORECASE) for p in config.fail_patterns]

    def classify(self, response: str) -> str:
        """
        Returns 'pass', 'fail', or 'unclear'.

        When both pass and fail patterns match (the model hedges or self-corrects),
        the label of the *last* match position is used as the final recommendation.
        This captures the model's concluding intent rather than an intermediate mention.
        """
        text = re.sub(r"[*_`]", "", response)  # strip markdown formatting
        pass_matches = [m for r in self._pass for m in r.finditer(text)]
        fail_matches = [m for r in self._fail for m in r.finditer(text)]

        has_pass = len(pass_matches) > 0
        has_fail = len(fail_matches) > 0

        if has_pass and not has_fail:
            return "pass"
        if has_fail and not has_pass:
            return "fail"
        if has_pass and has_fail:
            last_pass = max(m.start() for m in pass_matches)
            last_fail = max(m.start() for m in fail_matches)
            return "pass" if last_pass > last_fail else "fail"
        return "unclear"

    def score(self, responses: list) -> dict:
        """Scores a list of responses and returns accuracy + distribution."""
        labels = [self.classify(r) for r in responses]
        n_pass = sum(1 for l in labels if l == "pass")  # strict equality, not 'in'
        n_total = len(labels)
        return {
            "labels": labels,
            "n_pass": n_pass,
            "n_total": n_total,
            "accuracy": n_pass / n_total if n_total > 0 else 0,
            "distribution": dict(Counter(labels)),
        }


# Instantiate for the Car Wash Problem
config = EvalConfig()
evaluator = RegexEvaluator(config)
print("RegexEvaluator ready.")
print("Pass patterns :")
for p in config.pass_patterns:
    clean_p = p.replace("\\b", "")
    print(f"  - {clean_p}")

print("\nFail patterns :")
for p in config.fail_patterns:
    clean_p = p.replace("\\b", "")
    print(f"  - {clean_p}")

RegexEvaluator ready.
Pass patterns :
  - drive
  - driving
  - take the car
  - take your car
  - drive the car
  - car to the wash

Fail patterns :
  - walk
  - walking
  - on foot


## 6. Run Trials

In [7]:
def run_trial(system_prompt: str, question: str) -> str:
    """Send a single question to the model and return the response text."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        temperature=0.7,  # Non-zero temperature to measure probabilistic pass rate across trials
        system=system_prompt,
        messages=[{"role": "user", "content": question}],
    )
    return response.content[0].text

In [8]:
print(f"Running Condition C (STAR-only) × {n_trials} trials...")
condition_c_responses = []
for i in range(n_trials):
    resp = run_trial(STAR_ONLY_SYSTEM, QUESTION)
    condition_c_responses.append(resp)
    print(f"  [{i+1}/{n_trials}]")

print("\n" + "=" * 60)
print("Example response (Condition C):")
print("=" * 60)
print(condition_c_responses[0])

Running Condition C (STAR-only) × 20 trials...


  [1/20]


  [2/20]


  [3/20]


  [4/20]


...

  [16/20]


  [17/20]


  [18/20]


  [19/20]


  [20/20]

Example response (Condition C):
# Should You Walk or Drive to the Car Wash?

## Situation
You need to get your car washed at a facility 100 meters away — an extremely short distance.

## Task
Get your car to the car wash as efficiently and practically as possible.

## Action
**Drive your car to the car wash.**

Here's why this is clearly the right choice:

- 🚗 **The car needs to BE there** — You can't wash a car that isn't at the car wash
- 🚶 **Walking defeats the purpose** — You'd arrive without the car that needs washing
- 📏 **100 meters is trivial to drive** — roughly one city block, taking under a minute

## Result
Your car arrives at the car wash and gets cleaned. Problem solved.

---

## The Real Consideration
The **only practical question** is whether it's worth driving such a short distance *for you* afterward:
- You could **walk back home** while the car is being washed (100m is a 1-2 minute walk)
- Or **wait at the car wash**

**Bottom line: Drive the car there. Th

In [9]:
print(f"Running Condition A (Production) × {n_trials} trials...")
condition_a_responses = []
for i in range(n_trials):
    resp = run_trial(PRODUCTION_SYSTEM, QUESTION)
    condition_a_responses.append(resp)
    print(f"  [{i+1}/{n_trials}]")

print("\n" + "=" * 60)
print("Example response (Condition A):")
print("=" * 60)
print(condition_a_responses[0])

Running Condition A (Production) × 20 trials...


  [1/20]


  [2/20]


  [3/20]


  [4/20]


...

  [16/20]


  [17/20]


  [18/20]


  [19/20]


  [20/20]

Example response (Condition A):
That's outside the scope of a software engineering interview, but I'll bite — drive, since you need the car there anyway for the wash!

Is there a technical question I can help you with? I'm ready to discuss algorithms, system design, or anything relevant to the Software Engineer role at Google.


## 7. Results

In [10]:
scores_c = evaluator.score(condition_c_responses)
scores_a = evaluator.score(condition_a_responses)

print("Condition C (STAR-only):")
print(f"  Accuracy : {scores_c['accuracy']:.0%} ({scores_c['n_pass']}/{scores_c['n_total']})")
print(f"  Distribution: {scores_c['distribution']}")

print(f"\nCondition A (Production):")
print(f"  Accuracy : {scores_a['accuracy']:.0%} ({scores_a['n_pass']}/{scores_a['n_total']})")
print(f"  Distribution: {scores_a['distribution']}")

print(f"\nDilution Δ : {scores_c['accuracy'] - scores_a['accuracy']:.0%}")

Condition C (STAR-only):
  Accuracy : 90% (18/20)
  Distribution: {'pass': 18, 'fail': 2}

Condition A (Production):
  Accuracy : 30% (6/20)
  Distribution: {'pass': 6, 'fail': 13, 'unclear': 1}

Dilution Δ : 60%


In [11]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Accuracy comparison
    conditions = ["C: STAR-only", "A: Production\n(InterviewMate)"]
    accuracies = [scores_c["accuracy"] * 100, scores_a["accuracy"] * 100]
    colors = ["#2ecc71", "#e74c3c"]

    bars = axes[0].bar(conditions, accuracies, color=colors, width=0.5, edgecolor="black")
    axes[0].set_ylabel("Accuracy (%)")
    axes[0].set_title("Instruction Dilution Effect")
    axes[0].set_ylim(0, 110)
    for bar, acc in zip(bars, accuracies):
        axes[0].text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f"{acc:.0f}%", ha="center", va="bottom", fontweight="bold",
        )

    # Response distribution for Condition A
    dist_a = scores_a["distribution"]
    label_colors = {"pass": "#2ecc71", "fail": "#e74c3c", "unclear": "#95a5a6"}
    bar_colors = [label_colors.get(l, "#cccccc") for l in dist_a.keys()]
    axes[1].bar(list(dist_a.keys()), list(dist_a.values()), color=bar_colors, edgecolor="black")
    axes[1].set_xlabel("Classification")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Condition A: Response Distribution")

    plt.tight_layout()
    plt.show()

except ImportError:
    print("matplotlib not installed. Install with: pip install matplotlib")
    print("\nACCURACY COMPARISON")
    print("=" * 40)
    print(f"  Condition C (STAR-only):  {scores_c['accuracy']:>6.0%}")
    print(f"  Condition A (Production): {scores_a['accuracy']:>6.0%}")
    print(f"  Δ (dilution effect):      {scores_c['accuracy'] - scores_a['accuracy']:>6.0%}")

matplotlib not installed. Install with: pip install matplotlib

ACCURACY COMPARISON
  Condition C (STAR-only):     90%
  Condition A (Production):    30%
  Δ (dilution effect):         60%


## 8. Trial Log

In [12]:
print(f"{'Trial':<8} {'Cond C':<15} {'Cond A':<15}")
print("-" * 38)
for i in range(n_trials):
    lbl_c = scores_c["labels"][i]
    lbl_a = scores_a["labels"][i]
    mark_c = "✓" if lbl_c == "pass" else "✗"
    mark_a = "✓" if lbl_a == "pass" else "✗"
    print(f"{i+1:<8} {lbl_c:<7} {mark_c:<7} {lbl_a:<7} {mark_a}")

Trial    Cond C          Cond A         
--------------------------------------
1        pass    ✓       pass    ✓
2        pass    ✓       fail    ✗
3        pass    ✓       unclear ✗
4        pass    ✓       pass    ✓
5        pass    ✓       pass    ✓
6        pass    ✓       fail    ✗
7        pass    ✓       fail    ✗
8        pass    ✓       pass    ✓
9        pass    ✓       fail    ✗
10       pass    ✓       fail    ✗
11       pass    ✓       pass    ✓
12       pass    ✓       fail    ✗
13       pass    ✓       fail    ✗
14       pass    ✓       fail    ✗
15       fail    ✗       fail    ✗
16       pass    ✓       fail    ✗
17       pass    ✓       fail    ✗
18       pass    ✓       fail    ✗
19       fail    ✗       fail    ✗
20       pass    ✓       pass    ✓


## 9. Takeaway

### Why this matters for production AI systems

1. **Validate in your actual deployment environment.** A reasoning technique that scores 100% in a clean test prompt may score near 0% in production. The gap between a 10-line prompt and a 60-line prompt is not a matter of degree — it is a qualitative change in how the model processes instructions.

2. **Measure the dilution delta.** Run the same test cases under both a minimal prompt and your full production prompt. The gap tells you how much signal your other instructions are costing you.

3. **Instruction order is a first-class design variable.** “Answer first, then explain” and “Reason first, then conclude” are not stylistic choices — they determine whether structured reasoning frameworks function at all. In autoregressive generation, a premature conclusion commits subsequent tokens to a path that STAR reasoning cannot override.

4. **Reduce prompt length where possible.** Every line competes for the model’s attention.

5. **Run enough trials.** Use ≥ 20 trials per condition for a signal, ≥ 50 for tighter confidence intervals. Use `temperature > 0` to capture the probabilistic pass rate — `temperature=0` collapses variance and makes multiple trials meaningless.

> **Your eval must match your deployment context.** Instruction dilution is not a model bug — it is a consequence of how attention operates over long contexts.

In [13]:
summary = {
    "model": MODEL,
    "n_trials": n_trials,
    "temperature": 0.7,
    "question": QUESTION,
    "correct_answer": CORRECT_ANSWER,
    "condition_c": {
        "label": "STAR-only (arXiv:2602.21814)",
        "accuracy": scores_c["accuracy"],
        "distribution": scores_c["distribution"],
    },
    "condition_a": {
        "label": "Production — InterviewMate (arXiv:2603.13351)",
        "accuracy": scores_a["accuracy"],
        "distribution": scores_a["distribution"],
    },
    "dilution_delta": scores_c["accuracy"] - scores_a["accuracy"],
}

print(json.dumps(summary, indent=2))

{
  "model": "claude-sonnet-4-6",
  "n_trials": 20,
  "temperature": 0.7,
  "question": "I want to wash my car. The car wash is 100 meters away. Should I walk or drive?",
  "correct_answer": "drive",
  "condition_c": {
    "label": "STAR-only (arXiv:2602.21814)",
    "accuracy": 0.9,
    "distribution": {
      "pass": 18,
      "fail": 2
    }
  },
  "condition_a": {
    "label": "Production \u2014 InterviewMate (arXiv:2603.13351)",
    "accuracy": 0.3,
    "distribution": {
      "pass": 6,
      "fail": 13,
      "unclear": 1
    }
  },
  "dilution_delta": 0.6000000000000001
}
